In [27]:
import ee
from shapely.geometry import box
import geopandas as gpd
from shapely.geometry import Point


ee.Initialize(project="sentinel-487715")

# Load roads
roads = gpd.read_file("/Users/miranda/Documents/GitHub/Sentinel-FYP/data/gis_osm_roads_free_1.shp")
roads = roads.to_crs("EPSG:4326")

# Use projected CRS for distance checks
roads_m = roads.to_crs("EPSG:32630")  # UTM zone 30N

In [28]:
TOL = 15
FALLBACK_BUFFER = 75

def endpoints(geom):
    coords = list(geom.coords)
    return Point(coords[0]), Point(coords[-1])

if "start_pt" not in roads_m.columns or "end_pt" not in roads_m.columns:
    roads_m["start_pt"] = roads_m.geometry.apply(lambda g: endpoints(g)[0])
    roads_m["end_pt"] = roads_m.geometry.apply(lambda g: endpoints(g)[1])

sindex = roads_m.sindex

def connected_neighbors(row_idx):
    row = roads_m.loc[row_idx]
    start_pt = row["start_pt"]
    end_pt = row["end_pt"]

    def query_neighbors(pt):
        buf = pt.buffer(TOL)
        idxs = list(sindex.intersection(buf.bounds))
        cand = roads_m.iloc[idxs]
        return cand[cand.geometry.distance(pt) <= TOL]

    start_neighbors = query_neighbors(start_pt)
    end_neighbors = query_neighbors(end_pt)

    start_neighbors = start_neighbors[start_neighbors.index != row_idx]
    end_neighbors = end_neighbors[end_neighbors.index != row_idx]

    return start_neighbors, end_neighbors

def nearby_majority_class(row_idx, buffer_m=FALLBACK_BUFFER):
    row = roads_m.loc[row_idx]
    buf = row.geometry.buffer(buffer_m)

    idxs = list(sindex.intersection(buf.bounds))
    candidates = roads_m.iloc[idxs]
    candidates = candidates[candidates.geometry.intersects(buf)]

    candidates = candidates[candidates.index != row_idx]
    candidates = candidates[candidates["fclass"].astype(str).str.lower() != "unclassified"]

    if candidates.empty:
        return None

    vals, counts = np.unique(candidates["fclass"], return_counts=True)
    top = counts.max()
    top_vals = vals[counts == top]
    if len(top_vals) == 1:
        return top_vals[0]
    return None

def count_reclassified(orig, current):
    o = orig.astype(str).str.lower()
    c = current.astype(str).str.lower()
    return ((o == "unclassified") & (c != "unclassified")).sum()


In [29]:
# ---------- STEP 1 + STEP 2 ----------
updated = roads_m.copy()
orig_fclass = roads_m["fclass"].copy()

for idx, row in updated.iterrows():
    if str(row["fclass"]).lower() != "unclassified":
        continue

    start_neighbors, end_neighbors = connected_neighbors(idx)
    start_classes = start_neighbors["fclass"].dropna().tolist()
    end_classes = end_neighbors["fclass"].dropna().tolist()

    # Step 1: one-end or both-end majority vote
    if len(start_classes) > 0 and len(end_classes) == 0:
        vals, counts = np.unique(start_classes, return_counts=True)
        updated.at[idx, "fclass"] = vals[counts.argmax()]
        continue
    if len(end_classes) > 0 and len(start_classes) == 0:
        vals, counts = np.unique(end_classes, return_counts=True)
        updated.at[idx, "fclass"] = vals[counts.argmax()]
        continue

    all_classes = start_classes + end_classes
    if len(all_classes) > 0:
        vals, counts = np.unique(all_classes, return_counts=True)
        top = counts.max()
        top_vals = vals[counts == top]
        if len(top_vals) == 1:
            updated.at[idx, "fclass"] = top_vals[0]
        continue

    # Step 2 fallback: buffer majority
    fallback_class = nearby_majority_class(idx, buffer_m=FALLBACK_BUFFER)
    if fallback_class is not None:
        updated.at[idx, "fclass"] = fallback_class

print("After Step 1+2 reclassified:", count_reclassified(orig_fclass, updated["fclass"]))


After Step 1+2 reclassified: 6219


In [38]:
# Drop extra geometry columns before saving
save_gdf = updated_cc.drop(columns=["start_pt", "end_pt"], errors="ignore").copy()
save_gdf = save_gdf.set_geometry("geometry")

output_path = "/Users/miranda/Documents/GitHub/Sentinel-FYP/data/roads_reclassified.shp"
save_gdf.to_crs("EPSG:4326").to_file(output_path)




In [33]:
# ---------- STEP 4 + STEP 5: Component majority w/ length-weighted tie-break ----------
import networkx as nx

def neighbors_by_endpoint(idx):
    row = roads_m.loc[idx]
    pts = [row["start_pt"], row["end_pt"]]
    neigh_idx = set()

    for pt in pts:
        buf = pt.buffer(TOL)
        cand_idx = list(sindex.intersection(buf.bounds))
        cand = roads_m.iloc[cand_idx]
        cand = cand[cand.geometry.distance(pt) <= TOL]
        for j in cand.index:
            if j != idx:
                neigh_idx.add(j)
    return neigh_idx

G = nx.Graph()
G.add_nodes_from(roads_m.index)

for idx in roads_m.index:
    for j in neighbors_by_endpoint(idx):
        G.add_edge(idx, j)

updated_cc = updated.copy()

for component in nx.connected_components(G):
    comp_idx = list(component)
    comp = updated_cc.loc[comp_idx]

    labeled = comp[comp["fclass"].astype(str).str.lower() != "unclassified"]
    if labeled.empty:
        continue

    # First: simple majority count
    vals, counts = np.unique(labeled["fclass"], return_counts=True)
    top = counts.max()
    top_vals = vals[counts == top]

    if len(top_vals) == 1:
        majority_class = top_vals[0]
    else:
        # Step 5: length-weighted tie-break
        # Sum segment lengths per class
        lengths = labeled.copy()
        lengths["len"] = lengths.geometry.length
        length_sum = lengths.groupby("fclass")["len"].sum()
        # Check if weighted winner is unique
        max_len = length_sum.max()
        winners = length_sum[length_sum == max_len].index.tolist()
        if len(winners) != 1:
            continue  # tie even after weighting
        majority_class = winners[0]

    # Assign to unclassified in this component
    mask = comp["fclass"].astype(str).str.lower() == "unclassified"
    updated_cc.loc[comp.index[mask], "fclass"] = majority_class

print("After Step 4+5 reclassified:", count_reclassified(orig_fclass, updated_cc["fclass"]))



After Step 4+5 reclassified: 22957


In [34]:
# ---------- totals ----------
final_reclassified = count_reclassified(orig_fclass, updated_cc["fclass"])
final_remaining = (updated_cc["fclass"].astype(str).str.lower() == "unclassified").sum()

print("Total reclassified:", final_reclassified)
print("Still unclassified:", final_remaining)

Total reclassified: 22957
Still unclassified: 209


In [39]:
print("Original top 10 fclass:")
print(orig_fclass.value_counts().head(10))

print("\nReclassified top 10 fclass:")
print(updated_cc["fclass"].value_counts().head(10))


Original top 10 fclass:
fclass
residential     258612
service          36318
unclassified     23166
path             19722
track            17083
tertiary          4929
footway           4657
secondary         2516
trunk             1887
primary           1676
Name: count, dtype: int64

Reclassified top 10 fclass:
fclass
residential    277369
service         36402
path            19954
track           17374
tertiary         6680
footway          4712
secondary        3343
trunk            2358
primary          2148
trunk_link        482
Name: count, dtype: int64


In [40]:
# Count unclassified roads with a non-empty name
uncl = roads[roads["fclass"].astype(str).str.lower() == "unclassified"]

with_name = uncl["name"].notna() & (uncl["name"].astype(str).str.strip() != "")
print("Unclassified total:", len(uncl))
print("Unclassified with name:", with_name.sum())


Unclassified total: 23166
Unclassified with name: 1343
